In [1]:
import sys
print(sys.executable)

c:\Users\Alvar\anaconda3\envs\webscraping\python.exe


# Tarea 1 – Parte 1: Recolección y limpieza de noticias

**Fecha de descarga:** 17/10/2026  
**Medio elegido:** La República  
**Sitio web:** https://larepublica.pe/

## Categorías objetivo

La tarea considera cuatro categorías finales:

- `politica`
- `econonomia`
- `deportes`
- `ciencia_tecnologia`

Para respetar las secciones originales del medio, se utilizarán las siguientes secciones de La República:

- Política: https://larepublica.pe/politica
- Economía: https://larepublica.pe/economia
- Deportes: https://larepublica.pe/deportes
- Ciencia: https://larepublica.pe/ciencia

La sección original **Ciencia** será posteriormente normalizada a la etiqueta final `ciencia_tecnologia`.

In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from pathlib import Path

## 2. Prueba de conexión con La República

Antes de realizar la recolección masiva, se verifica que sea posible acceder correctamente a una de las secciones del medio mediante una solicitud HTTP.

In [5]:
url = "https://larepublica.pe/politica"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers, timeout=15)

print("Código de estado:", response.status_code)

Código de estado: 200


In [6]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

Últimas Noticias de Política del Perú en La República


In [7]:
for encabezado in soup.find_all(["h2", "h3"])[:15]:
    print(encabezado.get_text(strip=True))

Fracasó estrategia del estudio de abogados al que López Aliaga pagó S/97.5 millones
Solo el 28.3% aprueba la gestión de Keiko Fujimori
Predictamen de la Comisión de Constitución busca incorporar 8 puntos nuevos de las facultades aprobadas
Asesinato de Mauricio Ruiz Sanz ‘Trvko’ cumple un año sin resolverse
Detienen en Venezuela a cabecilla de la banda que mató a “China Polo”
Más noticias
Jaime Bayly: “Los políticos peruanos, sean de izquierda o de derecha, siempre encuentran la manera de decepcionarte”
Alejandro Aguinaga viajó a Argentina el martes 13 de octubre y todavía no regresa pese a debate por facultades
Álvaro Vargas Llosa niega intromisión política de embajador Navarro y asegura que solo tiene "una voz más participativa"
Solicitan la destitución de Roberto Burneo por avalar candidatura de Rafael López Aliaga
Puno: alcalde de Quiaca muere tras caer a un barranco con la camioneta que conducía con destino Juliaca
Ministro Rafael Belaúnde propone que las FF.AA apoyen en lucha cont

## 3. Identificación de titulares y enlaces

Luego de verificar el acceso al sitio web, se explora la estructura HTML de la sección Política para identificar los titulares de las noticias y sus respectivas URL.

In [8]:
from urllib.parse import urljoin

for encabezado in soup.find_all(["h2", "h3"])[:15]:
    titular = encabezado.get_text(strip=True)
    
    enlace = encabezado.find("a")
    
    if enlace is None:
        enlace = encabezado.find_parent("a")
    
    if enlace and enlace.get("href"):
        url_noticia = urljoin("https://larepublica.pe", enlace.get("href"))
        
        print("TITULAR:", titular)
        print("URL:", url_noticia)
        print("-" * 80)

TITULAR: Fracasó estrategia del estudio de abogados al que López Aliaga pagó S/97.5 millones
URL: https://larepublica.pe/politica/2026/10/16/fracaso-estrategia-del-estudio-de-abogados-al-que-rafael-lopez-aliaga-pago-s975-millones-mml-boies-schiller-flexner-martin-de-luca-hnews-1042515
--------------------------------------------------------------------------------
TITULAR: Solo el 28.3% aprueba la gestión de Keiko Fujimori
URL: https://larepublica.pe/politica/2026/10/14/solo-el-283-aprueba-la-gestion-de-keiko-fujimori-su-aceptacion-cayo-145-en-un-mes-segun-bloomberg-hnews-911652
--------------------------------------------------------------------------------
TITULAR: Predictamen de la Comisión de Constitución busca incorporar 8 puntos nuevos de las facultades aprobadas
URL: https://larepublica.pe/politica/2026/10/14/predictamen-de-la-comision-de-constitucion-busca-incorporar-8-puntos-nuevos-de-las-facultades-aprobadas-por-diputados-hnews-182238
-----------------------------------------

## 4. Construcción preliminar del conjunto de noticias de Política

A partir de los titulares y enlaces identificados en el HTML, se construye una estructura de datos con las noticias encontradas. Se eliminan los registros duplicados utilizando la URL como identificador único.

In [9]:
noticias_politica = []

for encabezado in soup.find_all(["h2", "h3"]):
    titular = encabezado.get_text(strip=True)

    enlace = encabezado.find("a")

    if enlace is None:
        enlace = encabezado.find_parent("a")

    if enlace and enlace.get("href"):
        url_noticia = urljoin(
            "https://larepublica.pe",
            enlace.get("href")
        )

        # Solo conservar noticias pertenecientes a Política
        if "/politica/" in url_noticia:
            noticias_politica.append({
                "titular": titular,
                "medio": "La República",
                "seccion_original": "politica",
                "url": url_noticia
            })

df_politica = pd.DataFrame(noticias_politica)

# Eliminar noticias repetidas según su URL
df_politica = (
    df_politica
    .drop_duplicates(subset="url")
    .reset_index(drop=True)
)

print("Noticias únicas encontradas:", len(df_politica))

df_politica.head(10)

Noticias únicas encontradas: 16


,titular,medio,seccion_original,url
0,Fracasó estrategia del estudio de abogados al ...,La República,politica,https://larepublica.pe/politica/2026/10/16/fra...
1,Solo el 28.3% aprueba la gestión de Keiko Fuji...,La República,politica,https://larepublica.pe/politica/2026/10/14/sol...
2,Predictamen de la Comisión de Constitución bus...,La República,politica,https://larepublica.pe/politica/2026/10/14/pre...
3,Asesinato de Mauricio Ruiz Sanz ‘Trvko’ cumple...,La República,politica,https://larepublica.pe/politica/2026/10/14/ase...
4,Detienen en Venezuela a cabecilla de la banda ...,La República,politica,https://larepublica.pe/politica/2026/10/14/det...
5,"Jaime Bayly: “Los políticos peruanos, sean de ...",La República,politica,https://larepublica.pe/politica/2026/02/14/jai...
6,Alejandro Aguinaga viajó a Argentina el martes...,La República,politica,https://larepublica.pe/politica/2026/10/16/ale...
7,Álvaro Vargas Llosa niega intromisión política...,La República,politica,https://larepublica.pe/politica/2026/10/16/alv...
8,Solicitan la destitución de Roberto Burneo por...,La República,politica,https://larepublica.pe/politica/2026/10/16/sol...
9,Puno: alcalde de Quiaca muere tras caer a un b...,La República,politica,https://larepublica.pe/politica/2026/10/16/pun...


## 5. Exploración de noticias anteriores

La página principal de Política contiene un número limitado de noticias visibles. Para alcanzar aproximadamente 90–100 noticias únicas, se revisará cómo el sitio permite acceder a publicaciones anteriores.

In [14]:
for elemento in soup.find_all(["a", "button"]):
    texto = elemento.get_text(" ", strip=True)

    if texto.lower() == "ver más noticias":
        print(elemento)

<button class="undefined ShowMoreButton_showMore__button__VBgwt">Ver más noticias<svg aria-hidden="true" class="ShowMoreButton_showMore__icon__Ni7U1" fill="none" height="20" viewbox="0 0 24 24" width="20"><path d="M6 9l6 6 6-6" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2"></path></svg></button>


## 6. Identificación del endpoint de paginación

Se identificó que el botón "Ver más noticias" de La República realiza solicitudes a un endpoint interno para cargar noticias adicionales.

Para la sección Política se utiliza el endpoint:

`https://larepublica.pe/api/search/articles`

La solicitud admite parámetros de categoría, cantidad de registros y número de página, lo que permitirá automatizar la recolección de noticias anteriores.

In [16]:
session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/142.0.0.0 Safari/537.36",
    "Referer": "https://larepublica.pe/politica",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-PE,es;q=0.9,en;q=0.8"
}

# Primero visitar la sección para obtener cookies
session.get(
    "https://larepublica.pe/politica",
    headers=headers,
    timeout=15
)

# Luego consultar la API
response_api = session.get(
    api_url,
    params=params,
    headers=headers,
    timeout=15
)

print("Código de estado:", response_api.status_code)
print("URL consultada:", response_api.url)

Código de estado: 200
URL consultada: https://larepublica.pe/api/search/articles?category_slug=politica&limit=12&page=3&order_by=update_date&view=section


In [17]:
print("Tipo de contenido:", response_api.headers.get("Content-Type"))

Tipo de contenido: application/json; charset=utf-8


In [18]:
datos_api = response_api.json()

print("Tipo de objeto:", type(datos_api))

Tipo de objeto: <class 'dict'>


In [19]:
if isinstance(datos_api, dict):
    print("Claves principales:")
    print(datos_api.keys())

Claves principales:
dict_keys(['articles'])


### Exploración de la respuesta de la API

La API devuelve la información dentro de la clave `articles`. A continuación, se revisa la cantidad de registros obtenidos y los campos disponibles para definir las variables que serán utilizadas en la recolección.

In [23]:
df_api = pd.json_normalize(articulos)

campos_utiles = [
    col for col in df_api.columns
    if any(palabra in col.lower() for palabra in
           ["title", "slug", "date", "description", "summary", "lead", "excerpt"])
]

print("Campos potencialmente útiles:")
print(campos_utiles)

df_api[campos_utiles].head()

Campos potencialmente útiles:
['title', 'date', 'updated_at', 'update_date', 'slug']


,title,date,updated_at,update_date,slug
0,Ministro Ernesto Álvarez descarta salida de la...,2026-10-15 08:12:39,2026-10-15 08:12:39,2026-10-15 08:12:39,/politica/2026/10/15/ministro-ernesto-alvarez-...
1,Detienen en Venezuela a cabecilla de la banda ...,2026-10-14 22:02:48,2026-10-15 02:22:52,2026-10-15 06:50:00,/politica/2026/10/14/detienen-en-venezuela-a-c...
2,Defensoría del Pueblo denuncia afectación al p...,2026-10-15 05:13:00,2026-10-15 01:14:25,2026-10-15 05:13:00,/politica/2026/10/15/defensoria-del-pueblo-den...
3,Raúl Tola: “Con los antecedentes del fujimoris...,2026-08-21 23:33:38,2026-10-15 01:03:21,2026-10-15 05:00:00,/politica/2026/08/21/raul-tola-con-los-anteced...
4,Rafael Rey confiesa que le ofrecieron la embaj...,2026-10-14 22:18:14,2026-10-14 22:18:14,2026-10-14 22:18:14,/politica/2026/10/14/rafael-rey-confiesa-que-l...


In [24]:
primer_articulo = articulos[0]

for campo in ["data", "metadata", "metadata_seo"]:
    valor = primer_articulo.get(campo)

    print(f"\n{campo}: {type(valor)}")

    if isinstance(valor, dict):
        print(valor.keys())
    else:
        print(valor)


data: <class 'dict'>
dict_keys(['teaser', 'authors', 'tags', 'categories', '__typename', 'multimedia'])

metadata: <class 'list'>
[{'key': 'censored', 'value': '0', '__typename': 'MetadataType'}, {'key': 'sponsored', 'value': None, '__typename': 'MetadataType'}, {'key': 'color', 'value': None, '__typename': 'MetadataType'}, {'key': 'logo', 'value': None, '__typename': 'MetadataType'}, {'key': 'recommended_video', 'value': '0', '__typename': 'MetadataType'}, {'key': 'advertorial', 'value': '0', '__typename': 'MetadataType'}, {'key': 'commercial_template', 'value': 'no', '__typename': 'MetadataType'}, {'key': 'deny_backlink_process', 'value': '0', '__typename': 'MetadataType'}]

metadata_seo: <class 'dict'>
dict_keys(['keywords', '__typename'])


In [25]:
teaser = articulos[0]["data"]["teaser"]

print("Tipo de teaser:", type(teaser))
print("Contenido:")
print(teaser)

Tipo de teaser: <class 'str'>
Contenido:
El ministro de Justicia señaló que una de las prioridades de su gestión es agilizar los procesos penales. Durante el último debate presidencial, Keiko Fujimori afirmó que evaluaría la posibilidad de retirar al Perú de la Corte IDH.


## 7. Recolección automatizada de noticias

A partir del endpoint identificado, se automatiza la consulta de páginas sucesivas de la sección, incorporando una pausa entre solicitudes. Los registros se estructuran con las variables requeridas y se eliminan duplicados por URL.

In [26]:
def recolectar_seccion(seccion, etiqueta, objetivo=100, max_paginas=15):
    registros = []
    
    for pagina in range(1, max_paginas + 1):
        params = {
            "category_slug": seccion,
            "limit": 12,
            "page": pagina,
            "order_by": "update_date",
            "view": "section"
        }

        respuesta = session.get(
            api_url,
            params=params,
            headers=headers,
            timeout=15
        )

        if respuesta.status_code != 200:
            print(f"Página {pagina}: error {respuesta.status_code}")
            continue

        articulos = respuesta.json()["articles"]["data"]

        if not articulos:
            break

        for articulo in articulos:
            slug = articulo.get("slug", "")
            datos = articulo.get("data") or {}

            registros.append({
                "titular": articulo.get("title", ""),
                "bajada": datos.get("teaser", ""),
                "medio": "La República",
                "seccion_original": seccion,
                "url": urljoin("https://larepublica.pe", slug),
                "fecha": articulo.get("date", ""),
                "etiqueta": etiqueta
            })

        # eliminar duplicados acumulados
        df_temp = pd.DataFrame(registros).drop_duplicates(subset="url")

        print(f"Página {pagina}: {len(df_temp)} noticias únicas")

        if len(df_temp) >= objetivo:
            break

        time.sleep(1.2)

    return (
        df_temp
        .drop_duplicates(subset="url")
        .query("titular != ''")
        .head(objetivo)
        .reset_index(drop=True)
    )

In [27]:
df_politica = recolectar_seccion(
    seccion="politica",
    etiqueta="politica",
    objetivo=100
)

print(f"\nTotal final de Política: {len(df_politica)}")
df_politica.head()

Página 1: 12 noticias únicas
Página 2: 24 noticias únicas
Página 3: 36 noticias únicas
Página 4: 48 noticias únicas
Página 5: 59 noticias únicas
Página 6: 71 noticias únicas
Página 7: 83 noticias únicas
Página 8: 95 noticias únicas
Página 9: 107 noticias únicas

Total final de Política: 100


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,''Evacúen sus casas desde ya'': Keiko Fujimori...,La mandataria mencionó que el fenómeno está af...,La República,politica,https://larepublica.pe/politica/2026/10/16/eva...,2026-10-16 07:52:00,politica
1,Keiko Fujimori sobre limpieza del río Piura an...,"Además, optó por darle la responsabilidad a la...",La República,politica,https://larepublica.pe/politica/2026/10/17/kei...,2026-10-17 08:36:31,politica
2,Debate de pedido de facultades sigue entrampad...,La Comisión de Constitución del Senado todavía...,La República,politica,https://larepublica.pe/politica/2026/10/17/deb...,2026-10-17 07:14:35,politica
3,Fuerza Popular desconoce votación que pedía de...,Cuatro bancadas de oposición abandonaron la se...,La República,politica,https://larepublica.pe/politica/2026/10/15/fue...,2026-10-15 19:05:12,politica
4,Solicitan la destitución de Roberto Burneo por...,"La asociación civil Iniciativa Democrática, or...",La República,politica,https://larepublica.pe/politica/2026/10/16/sol...,2026-10-16 14:54:58,politica
